In [2]:
# !pip install scikeras

# # Uninstall the current scikit-learn version
# !pip uninstall scikit-learn -y

# # Install a compatible version of scikit-learn (e.g., 1.4.2)
# !pip install scikit-learn==1.4.2

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from keras.callbacks import EarlyStopping

from scikeras.wrappers import KerasClassifier

from sklearn.calibration import CalibrationDisplay
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import SimpleImputer,IterativeImputer
from sklearn.preprocessing import LabelEncoder, StandardScaler,OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    accuracy_score,
    confusion_matrix,
    multilabel_confusion_matrix,
    f1_score
)
from sklearn.utils.class_weight import compute_class_weight

In [4]:
# Read training data
training_df = pd.read_csv(
    filepath_or_buffer='training_faults_diagnostics.csv',
    low_memory=False
)

In [5]:
# Convert SPN and FMI values to strings
training_df['spn'] = training_df['spn'].astype(str)
training_df['fmi'] = training_df['fmi'].astype(str)
print(f'spn datatype: {training_df['spn'].dtype}')

spn datatype: object


In [6]:
target = 'Derate_Target'

# Create dataset with features
X = training_df

# Create array of targets
y = training_df[target]

labels = np.unique(y)

## Identify features for imputing missing values

In [7]:
# Group categorical columns
categorical_columns = ['EquipmentID', 'spn', 'fmi', 'active', 'Severity_Level']

# Group numeric columns
numeric_columns = [
    'BarometricPressure',
    'EngineCoolantTemperature',
    'EngineLoad',
    'EngineOilPressure',
    'EngineOilTemperature',
    'EngineRpm',
    'FuelRate',
    'FuelTemperature',
    'IntakeManifoldTemperature',
    'Speed',
    'Throttle',
    'TurboBoostPressure'
  ]

## Split training dataset

In [8]:
random_state = 42

# Split training and testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=random_state,
    stratify=y
)

# Split training and validation
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.1,
    random_state=random_state,
    stratify=y
)

## Create pipeline and fit model

In [9]:
# Initialize pipeline for transforming categorical columns
categorical_pipe = Pipeline(
    steps=[
        ('categorical_imputer', SimpleImputer(strategy='most_frequent')),
        ('ohe', OneHotEncoder(handle_unknown='ignore'))
    ]
)

# Initialize pipeline for transforming numeric columns
numeric_pipe = Pipeline(
    steps=[
        ('numeric_imputer', IterativeImputer(max_iter=5, random_state=random_state)),
        ('scaler', StandardScaler())
    ]
)

In [10]:
# Initialize column transformer
ct = ColumnTransformer(
    transformers=[
        ('categorical_pipe', categorical_pipe, categorical_columns),
        ('numeric_pipe', numeric_pipe, numeric_columns)
    ]
)

In [11]:
# Get the number of features after one hot encoding
ct.fit(X_train)
X_val_transform = ct.transform(X_val)
n_features = ct.transform(X_train[:1]).shape[1]
print(f'n_features: {n_features}')

# Transform train labels
y_train_transform = tf.keras.utils.to_categorical(
    x=y_train.values,
    num_classes=len(labels)
)

# Transform validation labels
y_val_transform = tf.keras.utils.to_categorical(
    y_val.values,
    num_classes=len(labels)
)

# Transform test labels
y_test_transform = tf.keras.utils.to_categorical(
    x=y_test.values,
    num_classes=len(labels)
)

/usr/local/lib/python3.12/dist-packages/sklearn/impute/_iterative.py:801: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(


n_features: 1412


In [12]:
# Initialize early stopping
es = EarlyStopping(
    monitor='val_loss',
    patience=2,
    restore_best_weights=True
)

# Initialize class weights
# weights = compute_class_weight(
#     class_weight='balanced',
#     classes=labels,
#     y=y_train
# )
# class_weights = dict(zip(labels, weights))
# {0: 0.33392947189837874, 1: 397.27242386316226, 2: 352.29818719940806}

class_weights = {
    0: 1,
    1: 2,
    2: 1
}
class_weights

{0: 1, 1: 2, 2: 1}

In [13]:
# Function to create the Keras model for SciKeras
def create_model():
    model = tf.keras.Sequential()
    model.add(tf.keras.layers.InputLayer(shape=(n_features,)))
    model.add(tf.keras.layers.Dense(64, activation='relu'))
    model.add(tf.keras.layers.Dense(32, activation='relu'))
    model.add(tf.keras.layers.Dense(3, activation='softmax'))
    model.compile(
        optimizer='adam',
        loss='categorical_crossentropy',
        metrics=[
            tf.keras.metrics.Precision(),
            tf.keras.metrics.Recall()
        ]
    )
    return model

# Keras model with SciKeras wrapper
model = KerasClassifier(
    model=create_model,
    callbacks=[es],
    epochs=20,
    batch_size=32
)

In [14]:
# Initialize pipeline for training model
pipe = Pipeline(
    steps=[
        ('transformer', ct),
        ('model', model)
    ]
)

In [15]:
# Fit the model with training data, encoded labels, and validation data
pipe.fit(
    X=X_train,
    y=y_train_transform,
    model__validation_data=(X_val_transform, y_val_transform),
    model__class_weight=class_weights
)

/usr/local/lib/python3.12/dist-packages/sklearn/impute/_iterative.py:801: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(


Epoch 1/20
29759/29759 ━━━━━━━━━━━━━━━━━━━━ 73s 2ms/step - loss: 0.0128 - precision: 0.9986 - recall: 0.9977 - val_loss: 0.0086 - val_precision: 0.9986 - val_recall: 0.9985
Epoch 2/20
29759/29759 ━━━━━━━━━━━━━━━━━━━━ 65s 2ms/step - loss: 0.0104 - precision: 0.9986 - recall: 0.9984 - val_loss: 0.0075 - val_precision: 0.9986 - val_recall: 0.9984
Epoch 3/20
29759/29759 ━━━━━━━━━━━━━━━━━━━━ 64s 2ms/step - loss: 0.0100 - precision: 0.9986 - recall: 0.9984 - val_loss: 0.0073 - val_precision: 0.9986 - val_recall: 0.9983
Epoch 4/20
29759/29759 ━━━━━━━━━━━━━━━━━━━━ 65s 2ms/step - loss: 0.0099 - precision: 0.9986 - recall: 0.9984 - val_loss: 0.0070 - val_precision: 0.9986 - val_recall: 0.9983
Epoch 5/20
29759/29759 ━━━━━━━━━━━━━━━━━━━━ 64s 2ms/step - loss: 0.0097 - precision: 0.9986 - recall: 0.9984 - val_loss: 0.0077 - val_precision: 0.9986 - val_recall: 0.9982
Epoch 6/20
29759/29759 ━━━━━━━━━━━━━━━━━━━━ 64s 2ms/step - loss: 0.0097 - precision: 0.9986 - recall: 0.9984 - val_loss: 0.0079 - val_p

Pipeline(steps=[('transformer',
                 ColumnTransformer(transformers=[('categorical_pipe',
                                                  Pipeline(steps=[('categorical_imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('ohe',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['EquipmentID', 'spn', 'fmi',
                                                   'active',
                                                   'Severity_Level']),
                                                 ('numeric_pipe',
                                                  Pipeline(steps=[('numeric_imputer',
                                                                   IterativeImputer(max_iter=5,
                                                                                    rand...
                                                   'EngineLoad',
                                                   'EngineOilPressure',
                                                   'EngineOilTemperature',
                                                   'EngineRpm', 'FuelRate',
                                                   'FuelTemperature',
                                                   'IntakeManifoldTemperature',
                                                   'Speed', 'Throttle',
                                                   'TurboBoostPressure'])])),
                ('model',
                 KerasClassifier(batch_size=32, callbacks=[<keras.src.callbacks.early_stopping.EarlyStopping object at 0x7c0f19276030>], epochs=20, model=<function create_model at 0x7c0f19472340>))])

## Compare Training and Testing

In [17]:
# Predict training and testing data
y_pred_train = pipe.predict(X_train)
y_pred_test = pipe.predict(X_test)

29759/29759 ━━━━━━━━━━━━━━━━━━━━ 64s 2ms/step
6613/6613 ━━━━━━━━━━━━━━━━━━━━ 16s 2ms/step


In [21]:
X_train['y_pred_train'] = np.argmax(y_pred_train, axis=1)
X_test['y_pred_test'] = np.argmax(y_pred_test, axis=1)

In [22]:
X_train.to_csv('keras_pred_train.csv', index=False)
X_test.to_csv('keras_pred_test.csv', index=False)

In [27]:
# Training classification report
training_cr = classification_report(
    y_true=y_train,
    y_pred=X_train['y_pred_train'],
    digits=6
)
print(str(training_cr))

# Training confusion matrix
training_cm = confusion_matrix(
    y_true=y_train,
    y_pred=X_train['y_pred_train']
)
print(training_cm)

              precision    recall  f1-score   support

           0   0.998810  0.999822  0.999316    950562
           1   0.385455  0.132666  0.197393       799
           2   0.831169  0.426193  0.563463       901

    accuracy                       0.998552    952262
   macro avg   0.738478  0.519560  0.586724    952262
weighted avg   0.998137  0.998552  0.998231    952262

[[950393    128     41]
 [   656    106     37]
 [   476     41    384]]


In [30]:
# Testing classification report
testing_cr = classification_report(
    y_true=y_test,
    y_pred=X_test['y_pred_test'],
    digits=6
)
print(str(testing_cr))

# Testing confusion matrix
testing_cm = confusion_matrix(
    y_true=y_test,
    y_pred=X_test['y_pred_test']
)
print(testing_cm)

              precision    recall  f1-score   support

           0   0.998789  0.999830  0.999309    211236
           1   0.367647  0.140449  0.203252       178
           2   0.777778  0.350000  0.482759       200

    accuracy                       0.998493    211614
   macro avg   0.714738  0.496760  0.561773    211614
weighted avg   0.998050  0.998493  0.998151    211614

[[211200     27      9]
 [   142     25     11]
 [   114     16     70]]


## Predict Unseen Data

In [31]:
unseen_df = pd.read_csv(
    filepath_or_buffer='testing_faults_diagnostics.csv',
    low_memory=False
)

# Convert SPN and FMI values to strings
unseen_df['spn'] = unseen_df['spn'].astype(str)
unseen_df['fmi'] = unseen_df['fmi'].astype(str)
print(f'spn datatype: {unseen_df['spn'].dtype}')

y_unseen = unseen_df['Derate_Target']

spn datatype: object


In [32]:
# Transform unseen labels
y_unseen_transform = tf.keras.utils.to_categorical(
    x=y_unseen.values,
    num_classes=3
)

In [34]:
# Predict unseen data
y_pred_unseen = pipe.predict(unseen_df)

4040/4040 ━━━━━━━━━━━━━━━━━━━━ 10s 2ms/step


In [35]:
unseen_df['y_pred_unseen'] = np.argmax(y_pred_unseen, axis=1)

In [37]:
unseen_df.to_csv('keras_unseen_pred.csv', index=False)

In [36]:
# Unseen classification report
unseen_cr = classification_report(
    y_true=y_unseen,
    y_pred=unseen_df['y_pred_unseen'],
    digits=6
)
print(str(unseen_cr))

# Unseen confusion matrix
unseen_cm = confusion_matrix(
    y_true=y_unseen,
    y_pred=unseen_df['y_pred_unseen']
)
print(unseen_cm)

              precision    recall  f1-score   support

           0   0.998939  0.985190  0.992017    128970
           1   0.043301  0.431472  0.078704       197
           2   0.407407  0.444444  0.425121        99

    accuracy                       0.983932    129266
   macro avg   0.483216  0.620369  0.498614    129266
weighted avg   0.997029  0.983932  0.990191    129266

[[127060   1853     57]
 [   105     85      7]
 [    30     25     44]]
